In [1]:
# Usual imports
import secml
import numpy as np
from tqdm import tqdm
from scipy.special import softmax
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from joblib import Parallel, delayed
import pickle
import os
import pandas as pd
import csv
import shutil

# SecML
from secml.ml.features.normalization import CNormalizerMinMax
from secml.ml.peval.metrics import CMetricAccuracy
from secml.array import CArray
from secml.ml.classifiers import CClassifierPyTorch

# RobustBench
import robustbench
from robustbench.utils import load_model
from secml.utils import fm
from secml import settings

# Albi utils
from utils_attacks import *
from utils_CP import *

In [6]:
output_dir = fm.join(settings.SECML_MODELS_DIR, 'robustbench')


In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ProbRegressionWrapper(nn.Module):
    def __init__(self, classifier):
        super().__init__()
        self.classifier = classifier  # pretrained classifier

    def forward(self, x):
        logits = self.classifier(x)
        probs = F.softmax(logits, dim=1)
        return probs


In [9]:
model_wang = load_model('Wang2023Better_WRN-70-16', norm='L2', model_dir=output_dir)
regression_wang = ProbRegressionWrapper(model_wang)

reg_wang = CClassifierPyTorch(
    model=regression_wang,
    loss=nn.MSELoss(),
    optimizer=torch.optim.Adam(regression_wang.parameters(), lr=1e-4),
    input_shape=(3,32,32),
    softmax_outputs=False  # NOW REGRESSION, NOT CLASSIFICATION
)


In [41]:
import torch
import torchvision
import torchvision.transforms as T

# --------------------------
# Load CIFAR-10 test sample
# --------------------------
transform = T.Compose([
    T.ToTensor()
])

testset = torchvision.datasets.CIFAR10(
    root='./data', 
    train=False,
    download=True,
    transform=transform
)

img, label = testset[0]
x = img.unsqueeze(0)  # batch size = 1

# --------------------------
# Move to model device
# --------------------------
device = next(regression_wang.parameters()).device
x = x.to(device)

# --------------------------
# Predict probabilities
# --------------------------
regression_wang.eval()
with torch.no_grad():
    pred_probs = regression_wang(x)  # forward returns probabilities

# --------------------------
# Print results
# --------------------------
print("Predicted probabilities:", pred_probs.cpu().numpy())
print("Sum:", pred_probs.sum().item())       # should be ≈ 1.0
print("Predicted class:", pred_probs.argmax().item())
print("True label:", label)


Predicted probabilities: [[0.01575184 0.01080012 0.01442318 0.7155741  0.01017411 0.17864309
  0.01689676 0.01410748 0.01237364 0.01125565]]
Sum: 1.0
Predicted class: 3
True label: 3


In [53]:
print(x.shape)
print(test_X_t.shape)

torch.Size([1, 3, 32, 32])
torch.Size([2000, 3, 32, 32])


# PERCP ROBUSTBENCH - REGRESSION

In [94]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CUDA available: True
GPU name: NVIDIA GeForce RTX 4090


In [14]:
import torch
import numpy as np
from rich.progress import Progress

mse_loss = torch.nn.MSELoss()

def fgsm_attack(x, grad, epsilon=0.1):
    return x + epsilon * grad.sign()

def apply_fgsm(model, X, y, epsilon):
    """
    Pure PyTorch FGSM attack (works for regression or classification).
    X: tensor (B,3,32,32)
    y: tensor (B,num_classes) or regression target
    """
    # Clone and require gradients
    X_adv = X.clone().detach().requires_grad_(True)

    # Forward
    preds = model(X_adv)

    # Loss: MSE for regression model
    loss = nn.MSELoss()(preds, y)

    # Backward
    model.zero_grad()
    loss.backward()

    # FGSM step
    X_adv = X_adv + epsilon * X_adv.grad.sign()

    # Clamp to valid image range
    X_adv = torch.clamp(X_adv, 0, 1)

    return X_adv.detach()



### Just a quick try

In [15]:
# --------------------------
# Load CIFAR-10
# --------------------------
transform = T.Compose([
    T.ToTensor()
])

trainset = torchvision.datasets.CIFAR10(
    root='./data', 
    train=True,
    download=True,
    transform=transform
)

testset = torchvision.datasets.CIFAR10(
    root='./data', 
    train=False,
    download=True,
    transform=transform
)

In [18]:

img, label = testset[10]
x = img.unsqueeze(0).cuda()

y_onehot = torch.zeros(1, 10).cuda()
y_onehot[0, label] = 1.0

x_adv = apply_fgsm(regression_wang, x, y_onehot, 0.1)

print("Target probaility: ", label)
print("Clean x: ", regression_wang(x))
print("Attacked x: ", regression_wang(x_adv))


Target probaility:  0
Clean x:  tensor([[0.8442, 0.0103, 0.0223, 0.0294, 0.0123, 0.0265, 0.0155, 0.0117, 0.0171,
         0.0106]], device='cuda:0', grad_fn=<SoftmaxBackward0>)
Attacked x:  tensor([[0.0640, 0.0158, 0.1304, 0.1723, 0.2851, 0.1238, 0.1406, 0.0253, 0.0268,
         0.0159]], device='cuda:0', grad_fn=<SoftmaxBackward0>)


In [19]:
import torch
import torchvision
import torchvision.transforms as T
import numpy as np
from secml.array import CArray

# ------------------------------------------------------------
# Function: split CIFAR-10 into train / calibration / test
# ------------------------------------------------------------
def load_cifar10_splits(
    n_train=10000, 
    n_cal=2000, 
    n_test=5000,
    to_carray=True        # return as SECML CArray
):
    """
    Load CIFAR-10 and split into Train / Calibration / Test sets.

    Args:
        n_train: number of training samples
        n_cal: number of calibration samples
        n_test: number of test samples
        to_carray: if True return SECML CArray, else numpy arrays

    Returns:
        (X_train, y_train), (X_cal, y_cal), (X_test, y_test)
    """

    # --------------------------------------------------------
    # Load CIFAR-10
    # --------------------------------------------------------
    transform = T.Compose([T.ToTensor()])

    trainset = torchvision.datasets.CIFAR10(
        root='./data', train=True, download=True, transform=transform
    )

    testset = torchvision.datasets.CIFAR10(
        root='./data', train=False, download=True, transform=transform
    )

    # Convert datasets to tensors
    X_train_full = torch.stack([trainset[i][0] for i in range(len(trainset))])
    y_train_full = torch.tensor([trainset[i][1] for i in range(len(trainset))])

    X_test_full = torch.stack([testset[i][0] for i in range(len(testset))])
    y_test_full = torch.tensor([testset[i][1] for i in range(len(testset))])

    # --------------------------------------------------------
    # Random selection indices
    # --------------------------------------------------------
    total_train = len(X_train_full)
    perm = torch.randperm(total_train)

    if n_train + n_cal > total_train:
        raise ValueError("n_train + n_cal exceeds available CIFAR-10 training data!")

    idx_train = perm[:n_train]
    idx_cal = perm[n_train:n_train+n_cal]

    # Test set selection
    total_test = len(X_test_full)
    perm_test = torch.randperm(total_test)

    if n_test > total_test:
        raise ValueError("n_test exceeds available CIFAR-10 test data!")

    idx_test = perm_test[:n_test]

    # --------------------------------------------------------
    # Extract subsets
    # --------------------------------------------------------
    X_train = X_train_full[idx_train]
    y_train = y_train_full[idx_train]

    X_cal = X_train_full[idx_cal]   # calibration ALWAYS taken from training set
    y_cal = y_train_full[idx_cal]

    X_test = X_test_full[idx_test]
    y_test = y_test_full[idx_test]

    # --------------------------------------------------------
    # Convert to numpy / CArray if needed
    # --------------------------------------------------------
    def convert(X, y):
        X_np = X.numpy()
        y_np = y.numpy()
        if to_carray:
            return CArray(X_np), CArray(y_np)
        else:
            return X_np, y_np

    return (
        convert(X_train, y_train),
        convert(X_cal, y_cal),
        convert(X_test, y_test)
    )


In [69]:
def load_cifar10_splits_fast(
    n_train=1000,
    n_cal=500,
    n_test=2000
):
    transform = T.ToTensor()

    # Load datasets
    trainset = torchvision.datasets.CIFAR10(
        root='./data', train=True, download=True, transform=transform
    )
    testset = torchvision.datasets.CIFAR10(
        root='./data', train=False, download=True, transform=transform
    )

    # Random selection (only indices)
    perm_train = torch.randperm(len(trainset))
    perm_test = torch.randperm(len(testset))

    idx_train = perm_train[:n_train]
    idx_cal = perm_train[n_train:n_train+n_cal]
    idx_test = perm_test[:n_test]

    # Extract only needed samples (NO full dataset stacking)
    X_train = torch.stack([trainset[i][0] for i in idx_train])
    y_train = torch.tensor([trainset[i][1] for i in idx_train])

    X_cal = torch.stack([trainset[i][0] for i in idx_cal])
    y_cal = torch.tensor([trainset[i][1] for i in idx_cal])

    X_test = torch.stack([testset[i][0] for i in idx_test])
    y_test = torch.tensor([testset[i][1] for i in idx_test])

    return (X_train, y_train), (X_cal, y_cal), (X_test, y_test)

In [72]:
(train_X, train_y), (cal_X, cal_y), (test_X, test_y) = load_cifar10_splits_fast(
    n_train=1000,
    n_cal=500,
    n_test=2000)


In [73]:
train_X

tensor([[[[0.2667, 0.1804, 0.1490,  ..., 0.1294, 0.1294, 0.1294],
          [0.2078, 0.1922, 0.1686,  ..., 0.1216, 0.1294, 0.1255],
          [0.1412, 0.1255, 0.0902,  ..., 0.1412, 0.1412, 0.1373],
          ...,
          [0.3490, 0.2902, 0.3961,  ..., 0.4549, 0.5412, 0.5686],
          [0.4941, 0.3373, 0.3020,  ..., 0.5294, 0.5961, 0.6039],
          [0.8392, 0.7765, 0.6980,  ..., 0.8078, 0.8235, 0.8157]],

         [[0.2431, 0.1569, 0.1294,  ..., 0.1373, 0.1373, 0.1373],
          [0.1961, 0.1843, 0.1608,  ..., 0.1294, 0.1373, 0.1333],
          [0.1412, 0.1216, 0.0863,  ..., 0.1490, 0.1490, 0.1451],
          ...,
          [0.2902, 0.2235, 0.3176,  ..., 0.3804, 0.4706, 0.5020],
          [0.4510, 0.2824, 0.2392,  ..., 0.4784, 0.5451, 0.5529],
          [0.8196, 0.7569, 0.6745,  ..., 0.7843, 0.8000, 0.7922]],

         [[0.2275, 0.1333, 0.0941,  ..., 0.1216, 0.1294, 0.1333],
          [0.1765, 0.1569, 0.1294,  ..., 0.1137, 0.1294, 0.1294],
          [0.1098, 0.0941, 0.0667,  ..., 0

In [54]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ProbRegressionWrapper(nn.Module):
    def __init__(self, classifier):
        super().__init__()
        self.classifier = classifier  # pretrained classifier

    def forward(self, x):
        logits = self.classifier(x)
        probs = F.softmax(logits, dim=1)
        return probs


In [103]:
# 1. Load RobustBench
from robustbench import load_model
model_wang = load_model('Andriushchenko2020Understanding', norm='Linf', model_dir=output_dir).to(device)
#Wang2023Better_WRN-70-16
#Andriushchenko2020Understanding
# 2. Wrap as regression
regression_wang = ProbRegressionWrapper(model_wang)

# 3. Load CIFAR splits
(train_X, train_y), (cal_X, cal_y), (test_X, test_y) = load_cifar10_splits_fast(
    n_train=100, n_cal=100, n_test=1000
)

train_X = train_X.to(device)
train_y = train_y.to(device)

cal_X = cal_X.to(device)
cal_y = cal_y.to(device)

test_X = test_X.to(device)
test_y = test_y.to(device)

Downloading...
From: https://drive.google.com/uc?id=1Uyvprd98bIyxfMjLdCZwm-NEJ-6GMVis
To: /home/acarlevaro/secml-data/models/robustbench/cifar10/Linf/Andriushchenko2020Understanding.pt
100%|██████████| 89.5M/89.5M [00:02<00:00, 42.4MB/s]


In [104]:

# 4. One-hot encode
num_classes = 10
train_y_oh = F.one_hot(train_y, num_classes=num_classes).float()
cal_y_oh = F.one_hot(cal_y, num_classes=num_classes).float()
test_y_oh = F.one_hot(test_y, num_classes=num_classes).float()



In [105]:
# -------------------------------
# 6. Forward pass and regression MSE
# -------------------------------
with torch.no_grad():
    probs = regression_wang(test_X)
    print(probs)
    correct_probs = probs[range(len(test_y)), test_y]
    mse = ((correct_probs - 1) ** 2).mean()
    print("MSE on test set:", mse.item())

tensor([[1.6355e-02, 5.3831e-02, 1.6066e-04,  ..., 4.6569e-04, 1.6555e-03,
         9.2654e-01],
        [3.5299e-01, 4.5981e-05, 5.8888e-03,  ..., 2.1906e-03, 6.2519e-01,
         7.3059e-03],
        [4.2915e-03, 8.3462e-05, 4.1825e-04,  ..., 5.4057e-06, 9.9347e-01,
         2.5779e-04],
        ...,
        [6.7952e-01, 3.6035e-04, 2.4689e-01,  ..., 5.0568e-04, 2.9239e-02,
         5.3041e-04],
        [3.2093e-03, 6.7854e-04, 8.8480e-02,  ..., 1.4702e-04, 1.7946e-05,
         1.0012e-04],
        [1.6482e-04, 7.8129e-05, 1.2261e-02,  ..., 2.4782e-05, 1.1823e-04,
         2.4881e-05]], device='cuda:0')
MSE on test set: 0.2435191124677658


CUDA available: True
GPU name: NVIDIA GeForce RTX 4090
